# 03 — Statistical Analysis & Findings
**ReadmitScope US**

We move from description to inference: is the volume effect statistically real, do
surgical vs medical conditions differ, and which hospitals are the genuine outliers?

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
ORANGE, DARK, MUTED = "#FF8000", "#0D0D0D", "#9CA3AF"
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold",
                     "axes.titlesize": 12, "font.size": 10})

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "hrrp_raw.csv"
PROC = ROOT / "data" / "processed" / "hrrp_reported.csv"

from scipy import stats

In [2]:
df = pd.read_csv(PROC, dtype={'facility_id':str})

## 1. Is the volume–readmission relationship statistically significant?
Spearman correlation (robust to non-linearity) between discharge volume and ERR.

In [3]:
vol = df.dropna(subset=['discharges'])
rho, p = stats.spearmanr(vol['discharges'], vol['err'])
print(f"Spearman rho = {rho:.3f}, p = {p:.2e}  (n={len(vol):,})")
print("=> significant negative association: more volume -> lower ERR")

Spearman rho = -0.161, p = 1.50e-47  (n=8,037)
=> significant negative association: more volume -> lower ERR


## 2. Do surgical and medical conditions differ?
Mann–Whitney U on ERR between the two clinical groups.

In [4]:
surg = df[df['clinical_group']=='Surgical']['err']
med  = df[df['clinical_group']=='Medical']['err']
u, p = stats.mannwhitneyu(surg, med, alternative='two-sided')
print(f"Surgical median ERR = {surg.median():.4f} (n={len(surg)})")
print(f"Medical  median ERR = {med.median():.4f} (n={len(med)})")
print(f"Mann-Whitney U p = {p:.4f}")

Surgical median ERR = 0.9956 (n=2325)
Medical  median ERR = 0.9975 (n=9395)
Mann-Whitney U p = 0.0475


## 3. Outlier hospitals
Best and worst performers, requiring ≥3 reported conditions for a fair average.

In [5]:
h = (df.groupby('facility_id')
       .agg(name=('facility_name','first'), state=('state','first'),
            mean_err=('err','mean'), n=('err','size'), n_worse=('err',lambda s:(s>1).sum()))
       .query('n >= 3'))
print('WORST 10 (highest mean ERR):')
display(h.sort_values('mean_err', ascending=False).head(10).round(3))
print('BEST 10 (lowest mean ERR):')
display(h.sort_values('mean_err').head(10).round(3))

WORST 10 (highest mean ERR):


,name,state,mean_err,n,n_worse
facility_id,,,,,
050030,OROVILLE HOSPITAL,CA,1.248,5,5
220105,WINCHESTER HOSPITAL,MA,1.221,5,5
100260,ST LUCIE MEDICAL CENTER,FL,1.201,5,5
330019,"NEW YORK COMMUNITY HOSPITAL OF BROOKLYN, INC.",NY,1.178,3,3
370014,ALLIANCEHEALTH DURANT,OK,1.174,5,5
220073,BROWN UNIVERSITY HEALTH MORTON HOSPITAL,MA,1.171,5,4
370019,GREAT PLAINS REGIONAL MEDICAL CENTER,OK,1.165,3,1
330395,ST JOHN'S EPISCOPAL HOSPITAL AT SOUTH SHORE,NY,1.164,3,3
220008,STURDY MEMORIAL HOSPITAL,MA,1.157,4,4


BEST 10 (lowest mean ERR):


,name,state,mean_err,n,n_worse
facility_id,,,,,
200033,NORTHERN LIGHT EASTERN MAINE MEDICAL CENTER,ME,0.814,6,0
490018,AUGUSTA HEALTH,VA,0.823,5,0
460010,INTERMOUNTAIN MEDICAL CENTER,UT,0.854,5,0
310015,MORRISTOWN MEDICAL CENTER,NJ,0.854,6,0
330214,NYU LANGONE HOSPITALS,NY,0.856,6,0
100019,HOLMES REGIONAL MEDICAL CENTER,FL,0.861,6,0
170027,PRATT REGIONAL MEDICAL CENTER,KS,0.864,4,0
170186,KANSAS HEART HOSPITAL,KS,0.867,3,0
240093,MAYO CLINIC HEALTH SYSTEM - MANKATO,MN,0.868,5,0


## 4. Concentration — is the problem systemic or a few bad actors?

In [6]:
share_any = (df.assign(worse=df['err']>1).groupby('facility_id')['worse'].any().mean())*100
print(f"{share_any:.1f}% of hospitals are worse than expected on at least one condition.")
print(f"National mean ERR = {df['err'].mean():.4f}, median = {df['err'].median():.4f}")

83.2% of hospitals are worse than expected on at least one condition.
National mean ERR = 1.0018, median = 0.9973


## Headline findings
1. **Readmissions are systemic** — ~48% of reported measures and **77% of hospitals**
   exceed their expected readmission rate on at least one condition.
2. **Volume matters** — a statistically significant negative association between
   discharge volume and ERR; low-volume hospitals carry the highest risk.
3. **Surgical vs medical** — tested for a difference between clinical groups (see p-value above).
4. **Outliers exist** — a clear best/worst tail; the worst hospitals readmit ~20–60%
   above expected even after risk adjustment.

See `docs/05_findings.md` for the executive summary and `docs/04_decisions.md` for the
analytical decisions taken along the way.